# 02 — Thống kê về các streams

## 1. Phân tích toàn bộ stream công khai và đánh giá baseline

Nguồn: `scripts/analyze_public_streams.py`.

In [ ]:
"""Copy of the starter notebook extended to analyze every public stream.

This script keeps the same dataset assumptions as the notebook but adds a
stream-by-stream analysis pass for all public train/validation streams, then
writes a baseline validation submission and evaluates it.
"""

from __future__ import annotations

from pathlib import Path as _NotebookPath
_notebook_cwd = _NotebookPath.cwd().resolve()
_notebook_root = next((candidate for candidate in (_notebook_cwd, *_notebook_cwd.parents) if (candidate / 'scripts' / 'analyze_public_streams.py').is_file()), None)
if _notebook_root is None:
    raise FileNotFoundError('Could not locate the package root from the notebook working directory')
__file__ = str(_notebook_root / 'scripts' / 'analyze_public_streams.py')

import csv
import json
import sys
from pathlib import Path

import h5py
import matplotlib
import numpy as np

matplotlib.use("Agg")
import matplotlib.pyplot as plt

REPO = _notebook_root
ROOT = REPO

sys.path.insert(0, str(ROOT / "src"))
sys.path.insert(0, str(ROOT / "scripts"))

from abc2027.baseline import energy_events  # noqa: E402
from abc2027.evaluate import evaluate, load_ground_truth, stream_durations, validate_predictions  # noqa: E402

EVENT_STYLES = {
    "normal": ("tab:green", "normal"),
    "low": ("tab:orange", "low quality"),
    "needs_review": ("tab:red", "needs review"),
    "low_needs_review": ("tab:purple", "low quality + needs review"),
}


def stream_id_from_path(path: Path) -> str:
    """Map a stream_XX.h5 file to its public stream id, e.g. val_01."""
    stem = path.stem
    split = path.parent.name
    index = int(stem.split("_")[-1])
    return f"{split}_{index:02d}"


def event_rows_for_stream(rows: list[dict], stream_id: str) -> list[dict]:
    return [row for row in rows if row.get("stream_id") == stream_id]


def event_category(row: dict) -> str:
    low_quality = row.get("quality", "").strip().lower() == "low"
    needs_review = row.get("needs_review", "").strip().lower() in {"1", "true", "yes"}
    if low_quality and needs_review:
        return "low_needs_review"
    if low_quality:
        return "low"
    if needs_review:
        return "needs_review"
    return "normal"


def plot_stream_energy(stream_path: Path, event_rows: list[dict], output_dir: Path) -> Path:
    with h5py.File(stream_path, "r") as handle:
        cir = handle["cir"][:]
        t = handle["t"][:]

    energy = np.mean(np.sum(cir.astype(np.float64) ** 2, axis=2), axis=0)
    stream_id = stream_id_from_path(stream_path)
    output_dir.mkdir(parents=True, exist_ok=True)
    output_path = output_dir / f"{stream_id}_energy.png"

    figure, axis = plt.subplots(figsize=(16, 5))
    axis.plot(t[::10], energy[::10], color="black", linewidth=0.55, label="CIR energy")
    used_categories = set()
    for row in event_rows:
        category = event_category(row)
        color, label = EVENT_STYLES[category]
        axis.axvspan(
            float(row["t_start"]),
            float(row["t_end"]),
            color=color,
            alpha=0.3,
            label=label if category not in used_categories else None,
        )
        used_categories.add(category)

    axis.set_xlabel("Time (s)")
    axis.set_ylabel("Mean radar energy (mean sum of CIR²)")
    axis.set_title(f"{stream_id}: energy with annotated fall events (n={len(event_rows)})")
    axis.grid(True, alpha=0.2)
    axis.legend(loc="upper right")
    figure.tight_layout()
    figure.savefig(output_path, dpi=130)
    plt.close(figure)
    return output_path


def summarize_stream(stream_path: Path, event_rows: list[dict]) -> dict:
    with h5py.File(stream_path, "r") as handle:
        cir = handle["cir"][:]
        t = handle["t"][:]

    energy = np.mean(np.sum(cir.astype(np.float64) ** 2, axis=2), axis=0)
    categories = {category: 0 for category in EVENT_STYLES}
    for row in event_rows:
        categories[event_category(row)] += 1

    stats = {
        "stream_id": stream_id_from_path(stream_path),
        "path": str(stream_path),
        "duration_s": float(cir.shape[1] / 50.0),
        "n_frames": int(cir.shape[1]),
        "n_events": len(event_rows),
        "event_categories": categories,
        "energy_mean": float(np.mean(energy)),
        "energy_std": float(np.std(energy)),
        "energy_min": float(np.min(energy)),
        "energy_p90": float(np.percentile(energy, 90)),
        "energy_p95": float(np.percentile(energy, 95)),
        "energy_p99": float(np.percentile(energy, 99)),
        "max_energy": float(np.max(energy)),
        "events": [
            {
                "event_id": row["event_id"],
                "t_start": float(row["t_start"]),
                "t_end": float(row["t_end"]),
                "quality": row.get("quality", ""),
                "needs_review": row.get("needs_review", "").strip().lower() in {"1", "true", "yes"},
                "ignore": row.get("ignore", "").strip().lower() in {"1", "true", "yes"},
                "category": event_category(row),
            }
            for row in event_rows
        ],
    }
    return stats


def write_predictions_csv(path: Path, rows: list[dict]) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("w", encoding="utf-8", newline="") as handle:
        writer = csv.DictWriter(handle, fieldnames=["stream_id", "t_start", "t_end", "score", "t_alarm"])
        writer.writeheader()
        for row in rows:
            writer.writerow(
                {
                    "stream_id": row["stream_id"],
                    "t_start": f"{row['t_start']:.3f}",
                    "t_end": f"{row['t_end']:.3f}",
                    "score": f"{row['score']:.6f}",
                    "t_alarm": f"{row['t_alarm']:.3f}",
                }
            )


def analyze_all_streams() -> dict:
    events_path = ROOT / "streams_events.csv"
    with events_path.open(encoding="utf-8", newline="") as handle:
        all_events = list(csv.DictReader(handle))

    stats = []
    for split in ("train", "val"):
        folder = ROOT / "streams" / split
        for stream_path in sorted(folder.glob("stream_*.h5")):
            stream_id = stream_id_from_path(stream_path)
            gt_rows = event_rows_for_stream(all_events, stream_id)
            stream_stats = summarize_stream(stream_path, gt_rows)
            plot_path = plot_stream_energy(
                stream_path,
                gt_rows,
                ROOT / "outputs" / "stream_energy_plots",
            )
            stream_stats["energy_plot"] = str(plot_path)
            stats.append(stream_stats)

    return {"streams": stats, "n_streams": len(stats), "n_events": sum(item["n_events"] for item in stats)}


def build_baseline_submission(split: str = "val") -> list[dict]:
    folder = ROOT / "streams" / split
    rows: list[dict] = []
    for stream_path in sorted(folder.glob("stream_*.h5")):
        stream_id = stream_id_from_path(stream_path)
        with h5py.File(stream_path, "r") as handle:
            cir = handle["cir"][:]
        detected = energy_events(cir, percentile=98.0, min_duration=0.6, merge_gap=0.8)
        for event in detected:
            rows.append(
                {
                    "stream_id": stream_id,
                    "t_start": float(event["t_start"]),
                    "t_end": float(event["t_end"]),
                    "score": float(event["score"]),
                    "t_alarm": float(event["t_alarm"]),
                }
            )
    return rows


def main() -> None:
    print("Analyzing public streams...")
    analysis = analyze_all_streams()
    for item in analysis["streams"]:
        print(
            f"{item['stream_id']:>7} | duration={item['duration_s']:7.1f}s | "
            f"events={item['n_events']:>2} | normal={item['event_categories']['normal']:>2} "
            f"low={item['event_categories']['low']:>2} "
            f"review={item['event_categories']['needs_review'] + item['event_categories']['low_needs_review']:>2} | "
            f"energy_mean={item['energy_mean']:.4e} | energy_p99={item['energy_p99']:.4e}"
        )

    print(f"\nTotal public streams analyzed: {analysis['n_streams']}")
    print(f"Total annotated falls: {analysis['n_events']}")
    print(f"Energy plots saved under: {ROOT / 'outputs' / 'stream_energy_plots'}")

    val_rows = build_baseline_submission(split="val")
    out_path = ROOT / "outputs" / "all_val_baseline_submission_quality_analysis.csv"
    write_predictions_csv(out_path, val_rows)
    print(f"Saved baseline predictions to: {out_path}")

    durations = stream_durations(ROOT / "streams")
    gt = load_ground_truth(ROOT / "streams_events.csv", ignore_flagged=True)
    predictions = validate_predictions(out_path, durations)
    report = evaluate(gt, predictions, durations, theta=0.3)

    print("\nValidation report (ignore_flagged=True):")
    print(json.dumps({
        "ranking_score": report["ranking_score"],
        "primary_theta_0_3": report["primary"],
    }, indent=2))

    summary_path = ROOT / "outputs" / "stream_quality_analysis_summary.json"
    summary_path.write_text(json.dumps({"analysis": analysis, "report": report}, indent=2), encoding="utf-8")
    print(f"Saved summary JSON to: {summary_path}")


if __name__ == "__main__":
    main()


## 2. Trực quan hóa năng lượng CIR cùng các khoảng sự kiện ground-truth

Nguồn: `scripts/plot_stream_energy.py`.

In [ ]:
"""Plot stream CIR energy with ground-truth fall intervals."""

from __future__ import annotations

from pathlib import Path as _NotebookPath
_notebook_cwd = _NotebookPath.cwd().resolve()
_notebook_root = next((candidate for candidate in (_notebook_cwd, *_notebook_cwd.parents) if (candidate / 'scripts' / 'plot_stream_energy.py').is_file()), None)
if _notebook_root is None:
    raise FileNotFoundError('Could not locate the package root from the notebook working directory')
__file__ = str(_notebook_root / 'scripts' / 'plot_stream_energy.py')

import argparse
import csv
from collections import defaultdict
from pathlib import Path

import h5py
import matplotlib

matplotlib.use("Agg")
import matplotlib.pyplot as plt
from matplotlib.patches import Patch
import numpy as np

ROOT = Path(__file__).resolve().parents[1]
CHUNK_FRAMES = 2_000
PLOT_STEP = 10
EVENT_STYLES = {
    ("normal", False): ("#2ca02c", "normal"),
    ("normal", True): ("#d62728", "needs review"),
    ("low", False): ("#ff7f0e", "low quality"),
    ("low", True): ("#9467bd", "low quality + needs review"),
}


def load_events(path: Path) -> dict[str, list[dict[str, object]]]:
    events: dict[str, list[dict[str, object]]] = defaultdict(list)
    with path.open(encoding="utf-8", newline="") as handle:
        reader = csv.DictReader(handle)
        required = {"stream_id", "t_start", "t_end", "quality", "needs_review"}
        if not required.issubset(reader.fieldnames or ()):
            raise ValueError(f"{path} must contain columns: {', '.join(sorted(required))}")
        for row in reader:
            quality = row["quality"].strip().lower()
            needs_review = row["needs_review"].strip().lower() in {"1", "true", "yes"}
            if (quality, needs_review) not in EVENT_STYLES:
                raise ValueError(f"Unknown event quality/review flags: {quality!r}, {row['needs_review']!r}")
            events[row["stream_id"]].append(
                {
                    "t_start": float(row["t_start"]),
                    "t_end": float(row["t_end"]),
                    "quality": quality,
                    "needs_review": needs_review,
                }
            )
    return events


def stream_energy(path: Path) -> tuple[np.ndarray, np.ndarray]:
    with h5py.File(path, "r") as handle:
        cir = handle["cir"]
        times = handle["t"][:]
        if cir.ndim != 3 or len(times) != cir.shape[1]:
            raise ValueError(f"{path}: expected cir shaped (radars, time, bins) and matching t")

        energy = np.empty(cir.shape[1], dtype=np.float64)
        for start in range(0, cir.shape[1], CHUNK_FRAMES):
            stop = min(start + CHUNK_FRAMES, cir.shape[1])
            frames = cir[:, start:stop, :].astype(np.float64)
            energy[start:stop] = np.mean(np.sum(frames**2, axis=2), axis=0)
    return times, energy


def plot_stream(
    path: Path,
    stream_id: str,
    events: list[dict[str, object]],
    output_dir: Path,
) -> Path:
    times, energy = stream_energy(path)
    figure, axis = plt.subplots(figsize=(14, 4))
    axis.plot(times[::PLOT_STEP], energy[::PLOT_STEP], color="black", linewidth=0.65, label="CIR energy")

    for event in events:
        quality = str(event["quality"])
        needs_review = bool(event["needs_review"])
        status_color, _label = EVENT_STYLES[(quality, needs_review)]
        start = float(event["t_start"])
        end = float(event["t_end"])
        axis.axvspan(
            start,
            end,
            color="red",
            alpha=0.2,
            linewidth=0,
        )
        axis.axvline(start, color=status_color, linewidth=1.1, alpha=0.9)
        axis.axvline(end, color=status_color, linewidth=1.1, alpha=0.9)

    axis.set_xlabel("Time (s)")
    axis.set_ylabel("Mean radar energy (mean sum of CIR²)")
    axis.set_title(f"{stream_id}: CIR energy and ground-truth falls (n={len(events)})")
    axis.grid(True, alpha=0.25)
    if events:
        legend_handles = [
            plt.Line2D([], [], color="black", linewidth=0.8, label="CIR energy"),
            Patch(facecolor="red", edgecolor="none", alpha=0.2, label="Ground-truth fall interval"),
        ]
        legend_handles.extend(
            plt.Line2D([], [], color=color, linewidth=1.5, label=f"Event status: {label}")
            for color, label in EVENT_STYLES.values()
        )
        axis.legend(handles=legend_handles)
    figure.tight_layout()

    output_path = output_dir / f"{stream_id}_energy_ground_truth.png"
    figure.savefig(output_path, dpi=140)
    plt.close(figure)
    return output_path


def main() -> None:
    parser = argparse.ArgumentParser(description=__doc__)
    parser.add_argument(
        "--output-dir",
        type=Path,
        default=ROOT / "outputs" / "stream_energy_ground_truth",
        help="Directory for the generated PNG plots.",
    )
    args = parser.parse_args(args=[])

    events_by_stream = load_events(ROOT / "streams_events.csv")
    stream_paths = sorted((ROOT / "streams").glob("*/*.h5"))
    if not stream_paths:
        raise FileNotFoundError(f"No stream HDF5 files found under {ROOT / 'streams'}")

    args.output_dir.mkdir(parents=True, exist_ok=True)
    for path in stream_paths:
        stream_id = f"{path.parent.name}_{path.stem.rsplit('_', 1)[-1]}"
        output_path = plot_stream(
            path,
            stream_id,
            events_by_stream.get(stream_id, []),
            args.output_dir,
        )
        print(f"{stream_id}: {len(events_by_stream.get(stream_id, []))} events -> {output_path}")


if __name__ == "__main__":
    main()


## 3. Tổng hợp chất lượng sự kiện và cờ cần rà soát

Nguồn: `scripts/summarize_event_quality.py`.

In [ ]:
"""Summarize event quality/review labels and plot their per-stream composition."""

from __future__ import annotations

from pathlib import Path as _NotebookPath
_notebook_cwd = _NotebookPath.cwd().resolve()
_notebook_root = next((candidate for candidate in (_notebook_cwd, *_notebook_cwd.parents) if (candidate / 'scripts' / 'summarize_event_quality.py').is_file()), None)
if _notebook_root is None:
    raise FileNotFoundError('Could not locate the package root from the notebook working directory')
__file__ = str(_notebook_root / 'scripts' / 'summarize_event_quality.py')

import argparse
import csv
from collections import Counter, defaultdict
from pathlib import Path

import matplotlib

matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np

ROOT = Path(__file__).resolve().parents[1]
COMBINATIONS = (
    ("normal", False, "normal_no_review", "#2ca02c", "normal, no review"),
    ("normal", True, "normal_needs_review", "#d62728", "normal, needs review"),
    ("low", False, "low_no_review", "#ff7f0e", "low, no review"),
    ("low", True, "low_needs_review", "#9467bd", "low, needs review"),
)
CSV_FIELDS = (
    "scope",
    "split",
    "stream_id",
    "event_count",
    "quality_normal_count",
    "quality_normal_pct",
    "quality_low_count",
    "quality_low_pct",
    "needs_review_false_count",
    "needs_review_false_pct",
    "needs_review_true_count",
    "needs_review_true_pct",
    *(f"{name}_count" for _, _, name, _, _ in COMBINATIONS),
    *(f"{name}_pct" for _, _, name, _, _ in COMBINATIONS),
    "ignore_true_count",
    "ignore_true_pct",
)


def parse_bool(value: str, field: str, stream_id: str) -> bool:
    normalized = value.strip().lower()
    if normalized in {"true", "1", "yes"}:
        return True
    if normalized in {"false", "0", "no"}:
        return False
    raise ValueError(f"{stream_id}: invalid {field} value {value!r}")


def load_events(path: Path) -> dict[str, list[dict[str, object]]]:
    events: dict[str, list[dict[str, object]]] = defaultdict(list)
    required = {"stream_id", "quality", "needs_review", "ignore"}
    with path.open(encoding="utf-8", newline="") as handle:
        reader = csv.DictReader(handle)
        if not required.issubset(reader.fieldnames or ()):
            raise ValueError(f"{path} must contain columns: {', '.join(sorted(required))}")
        for row in reader:
            stream_id = row["stream_id"].strip()
            quality = row["quality"].strip().lower()
            if quality not in {"normal", "low"}:
                raise ValueError(f"{stream_id}: invalid quality value {row['quality']!r}")
            needs_review = parse_bool(row["needs_review"], "needs_review", stream_id)
            ignore = parse_bool(row["ignore"], "ignore", stream_id)
            events[stream_id].append(
                {"quality": quality, "needs_review": needs_review, "ignore": ignore}
            )
    if not events:
        raise ValueError(f"No event rows found in {path}")
    return events


def summarize(scope: str, split: str, stream_id: str, events: list[dict[str, object]]) -> dict[str, object]:
    count = len(events)
    quality_counts = Counter(str(event["quality"]) for event in events)
    review_counts = Counter(bool(event["needs_review"]) for event in events)
    combination_counts = Counter(
        (str(event["quality"]), bool(event["needs_review"])) for event in events
    )
    ignore_count = sum(bool(event["ignore"]) for event in events)

    def percentage(value: int) -> float:
        return round(100.0 * value / count, 2) if count else 0.0

    row: dict[str, object] = {
        "scope": scope,
        "split": split,
        "stream_id": stream_id,
        "event_count": count,
        "quality_normal_count": quality_counts["normal"],
        "quality_normal_pct": percentage(quality_counts["normal"]),
        "quality_low_count": quality_counts["low"],
        "quality_low_pct": percentage(quality_counts["low"]),
        "needs_review_false_count": review_counts[False],
        "needs_review_false_pct": percentage(review_counts[False]),
        "needs_review_true_count": review_counts[True],
        "needs_review_true_pct": percentage(review_counts[True]),
        "ignore_true_count": ignore_count,
        "ignore_true_pct": percentage(ignore_count),
    }
    for quality, needs_review, name, _, _ in COMBINATIONS:
        value = combination_counts[(quality, needs_review)]
        row[f"{name}_count"] = value
        row[f"{name}_pct"] = percentage(value)
    return row


def write_summary(path: Path, rows: list[dict[str, object]]) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("w", encoding="utf-8", newline="") as handle:
        writer = csv.DictWriter(handle, fieldnames=CSV_FIELDS)
        writer.writeheader()
        writer.writerows(rows)


def plot_composition(
    events_by_stream: dict[str, list[dict[str, object]]],
    path: Path,
) -> None:
    splits = ("train", "val")
    figure, axes = plt.subplots(2, 1, figsize=(14, 9), sharey=True)
    for axis, split in zip(axes, splits):
        stream_ids = sorted(
            stream_id for stream_id in events_by_stream if stream_id.startswith(f"{split}_")
        )
        if not stream_ids:
            axis.set_visible(False)
            continue

        bottom = np.zeros(len(stream_ids), dtype=np.float64)
        for quality, needs_review, name, color, label in COMBINATIONS:
            values = []
            for stream_id in stream_ids:
                events = events_by_stream[stream_id]
                matching = sum(
                    event["quality"] == quality and event["needs_review"] == needs_review
                    for event in events
                )
                values.append(100.0 * matching / len(events))
            axis.bar(
                stream_ids,
                values,
                bottom=bottom,
                color=color,
                label=label,
                width=0.78,
            )
            bottom += values

        for index, stream_id in enumerate(stream_ids):
            axis.text(
                index,
                102,
                f"n={len(events_by_stream[stream_id])}",
                ha="center",
                va="bottom",
                fontsize=8,
                rotation=45,
            )
        axis.set_title(f"{split}: event quality/review composition by stream")
        axis.set_ylabel("Events (%)")
        axis.set_ylim(0, 114)
        axis.set_xticks(range(len(stream_ids)), stream_ids, rotation=45, ha="right")
        axis.grid(axis="y", alpha=0.25)

    handles, labels = axes[0].get_legend_handles_labels()
    figure.legend(handles, labels, loc="upper center", bbox_to_anchor=(0.5, 0.99), ncols=4)
    axes[-1].set_xlabel("Stream ID")
    figure.subplots_adjust(top=0.91, bottom=0.1, hspace=0.48)
    path.parent.mkdir(parents=True, exist_ok=True)
    figure.savefig(path, dpi=150)
    plt.close(figure)


def main() -> None:
    parser = argparse.ArgumentParser(description=__doc__)
    parser.add_argument(
        "--output-dir",
        type=Path,
        default=ROOT / "outputs" / "quality_review_analysis",
        help="Directory for the summary CSV.",
    )
    parser.add_argument(
        "--plot",
        type=Path,
        default=ROOT / "outputs" / "event_quality_review_by_stream.png",
        help="Path for the 100% stacked bar chart.",
    )
    args = parser.parse_args(args=[])

    events_by_stream = load_events(ROOT / "streams_events.csv")
    streams_by_split: dict[str, list[dict[str, object]]] = defaultdict(list)
    for stream_id, events in events_by_stream.items():
        split = stream_id.split("_", 1)[0]
        if split not in {"train", "val"}:
            raise ValueError(f"Unexpected stream split in {stream_id!r}")
        streams_by_split[split].extend(events)

    rows = [summarize("overall", "all", "all", [event for group in streams_by_split.values() for event in group])]
    for split in ("train", "val"):
        rows.append(summarize("split", split, "", streams_by_split.get(split, [])))
        rows.extend(
            summarize("stream", split, stream_id, events)
            for stream_id, events in sorted(events_by_stream.items())
            if stream_id.startswith(f"{split}_")
        )

    csv_path = args.output_dir / "event_quality_review_summary.csv"
    write_summary(csv_path, rows)
    plot_composition(events_by_stream, args.plot)
    print(f"Wrote {len(rows)} summary rows to {csv_path}")
    print(f"Wrote 100% stacked chart to {args.plot}")


if __name__ == "__main__":
    main()


## 4. Thống kê số lượng, loại và thời lượng sự kiện ngã

Nguồn: `scripts/summarize_fall_events.py`.

In [ ]:
"""Summarize ground-truth fall events per stream and plot duration histograms."""

from __future__ import annotations

from pathlib import Path as _NotebookPath
_notebook_cwd = _NotebookPath.cwd().resolve()
_notebook_root = next((candidate for candidate in (_notebook_cwd, *_notebook_cwd.parents) if (candidate / 'scripts' / 'summarize_fall_events.py').is_file()), None)
if _notebook_root is None:
    raise FileNotFoundError('Could not locate the package root from the notebook working directory')
__file__ = str(_notebook_root / 'scripts' / 'summarize_fall_events.py')

import argparse
import csv
import math
from collections import defaultdict
from pathlib import Path

import matplotlib

matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np

ROOT = Path(__file__).resolve().parents[1]
FALL_TYPES = ("Chute01", "Chute02", "Chute03", "Chute04")
CSV_FIELDS = (
    "stream_id",
    "split",
    "event_count",
    *(f"{fall_type.lower()}_count" for fall_type in FALL_TYPES),
    "duration_mean_s",
    "duration_median_s",
    "duration_std_s",
)


def load_events(path: Path) -> dict[str, list[dict[str, str]]]:
    events: dict[str, list[dict[str, str]]] = defaultdict(list)
    required = {"stream_id", "t_start", "t_end", "fall_type"}
    with path.open(encoding="utf-8", newline="") as handle:
        reader = csv.DictReader(handle)
        if not required.issubset(reader.fieldnames or ()):
            raise ValueError(f"{path} must contain columns: {', '.join(sorted(required))}")
        for row in reader:
            events[row["stream_id"]].append(row)
    return events


def stream_ids() -> list[str]:
    paths = sorted((ROOT / "streams").glob("*/*.h5"))
    ids = [f"{path.parent.name}_{path.stem.rsplit('_', 1)[-1]}" for path in paths]
    if not ids:
        raise FileNotFoundError(f"No stream HDF5 files found under {ROOT / 'streams'}")
    return ids


def summarize_stream(stream_id: str, events: list[dict[str, str]]) -> dict[str, object]:
    durations = []
    counts = {fall_type: 0 for fall_type in FALL_TYPES}
    for event in events:
        start = float(event["t_start"])
        end = float(event["t_end"])
        duration = end - start
        if not math.isfinite(duration) or duration <= 0:
            raise ValueError(f"{stream_id}: invalid event interval {start} to {end}")
        durations.append(duration)
        fall_type = event["fall_type"]
        if fall_type not in counts:
            raise ValueError(f"{stream_id}: unknown fall_type {fall_type!r}")
        counts[fall_type] += 1

    row: dict[str, object] = {
        "stream_id": stream_id,
        "split": stream_id.split("_", 1)[0],
        "event_count": len(events),
    }
    row.update({f"{fall_type.lower()}_count": counts[fall_type] for fall_type in FALL_TYPES})
    if durations:
        row.update(
            {
                "duration_mean_s": round(float(np.mean(durations)), 3),
                "duration_median_s": round(float(np.median(durations)), 3),
                "duration_std_s": round(float(np.std(durations)), 3),
            }
        )
    else:
        row.update({"duration_mean_s": "", "duration_median_s": "", "duration_std_s": ""})
    return row


def write_summary(path: Path, rows: list[dict[str, object]]) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("w", encoding="utf-8", newline="") as handle:
        writer = csv.DictWriter(handle, fieldnames=CSV_FIELDS)
        writer.writeheader()
        writer.writerows(rows)


def plot_duration_histogram(events_by_stream: dict[str, list[dict[str, str]]], path: Path) -> None:
    durations_by_split: dict[str, list[float]] = defaultdict(list)
    for stream_id, events in events_by_stream.items():
        split = stream_id.split("_", 1)[0]
        for event in events:
            durations_by_split[split].append(float(event["t_end"]) - float(event["t_start"]))

    if not durations_by_split:
        raise ValueError("No ground-truth events found; cannot plot duration histogram")

    all_durations = [duration for values in durations_by_split.values() for duration in values]
    bins = np.histogram_bin_edges(all_durations, bins="fd")
    if len(bins) < 2:
        bins = np.linspace(min(all_durations) - 0.5, max(all_durations) + 0.5, 11)

    path.parent.mkdir(parents=True, exist_ok=True)
    figure, axis = plt.subplots(figsize=(9, 5))
    for split in ("train", "val"):
        durations = durations_by_split.get(split, [])
        if durations:
            axis.hist(durations, bins=bins, alpha=0.55, label=f"{split} (n={len(durations)})")
    axis.set_xlabel("Fall event duration (s)")
    axis.set_ylabel("Number of events")
    axis.set_title("Ground-truth fall duration distribution")
    axis.grid(axis="y", alpha=0.25)
    axis.legend()
    figure.tight_layout()
    figure.savefig(path, dpi=150)
    plt.close(figure)


def main() -> None:
    parser = argparse.ArgumentParser(description=__doc__)
    parser.add_argument(
        "--csv",
        type=Path,
        default=ROOT / "outputs" / "fall_event_summary_by_stream.csv",
        help="Output CSV path.",
    )
    parser.add_argument(
        "--histogram",
        type=Path,
        default=ROOT / "outputs" / "fall_duration_histogram.png",
        help="Output histogram PNG path.",
    )
    args = parser.parse_args(args=[])

    events_by_stream = load_events(ROOT / "streams_events.csv")
    rows = [
        summarize_stream(stream_id, events_by_stream.get(stream_id, []))
        for stream_id in stream_ids()
    ]
    write_summary(args.csv, rows)
    plot_duration_histogram(events_by_stream, args.histogram)
    print(f"Wrote {len(rows)} stream summaries to {args.csv}")
    print(f"Wrote duration histogram to {args.histogram}")


if __name__ == "__main__":
    main()
